# Turb-DETR Week 2A: Data Prep + SimAM Injection
**NO GPU needed.** Estimated: ~30-40 min

Does: augment training images, create combined dataset, patch Ultralytics with SimAM, save to Drive.

In [1]:
import os, time
t0 = time.time()
from google.colab import drive
drive.mount('/content/drive')

DATASET = "/content/drive/MyDrive/underwater_datasets/trash_ICRA19/dataset"
if not os.path.exists(DATASET):
    base = "/content/drive/MyDrive/underwater_datasets"
    for n in os.listdir(base):
        if "icra" in n.lower():
            for sub in ["dataset", "Dataset", ""]:
                cand = os.path.join(base, n, sub) if sub else os.path.join(base, n)
                if os.path.exists(os.path.join(cand, "train")):
                    DATASET = cand; break
assert os.path.exists(os.path.join(DATASET, "train"))
os.makedirs("/content/turb-detr", exist_ok=True)
os.chdir("/content/turb-detr")
print(f"Dataset: {DATASET} ({time.time()-t0:.1f}s)")

Mounted at /content/drive
Dataset: /content/drive/MyDrive/underwater_datasets/trash_ICRA19/dataset (32.0s)


In [2]:
import os, subprocess, time
from pathlib import Path
t0 = time.time()
SRC = Path(DATASET)
OUT = Path("/content/turb-detr/data/processed")
LOCAL = Path("/content/local_labels")

for split in ["train", "val", "test"]:
    src_dir = SRC / split
    img_out = OUT / split / "images"
    img_out.mkdir(parents=True, exist_ok=True)
    for f in src_dir.glob("*.jpg"):
        dst = img_out / f.name
        if not dst.exists(): os.symlink(f.resolve(), dst)

    local_lbl = LOCAL / split
    local_lbl.mkdir(parents=True, exist_ok=True)
    subprocess.run(f"cp {src_dir}/*.txt {local_lbl}/ 2>/dev/null", shell=True)

    lbl_out = OUT / split / "labels"
    lbl_out.mkdir(parents=True, exist_ok=True)
    rov = 0
    for txt in local_lbl.glob("*.txt"):
        lines = []
        for line in open(txt):
            parts = line.strip().split()
            if len(parts) == 5 and int(parts[0]) != 2:
                lines.append(line.strip())
            elif len(parts) == 5 and int(parts[0]) == 2:
                rov += 1
        with open(lbl_out / txt.name, "w") as f:
            f.write("\n".join(lines) + "\n" if lines else "")
    print(f"  {split}: {rov} ROV removed")

bad = OUT / "train" / "images" / "obj1658_frame0002383 (1).jpg"
if bad.exists() or bad.is_symlink(): os.unlink(bad)
print(f"Done ({time.time()-t0:.1f}s)")

  train: 1802 ROV removed
  val: 141 ROV removed
  test: 335 ROV removed
Done (3246.8s)


In [3]:
import cv2, numpy as np, shutil, time
from pathlib import Path
from tqdm import tqdm
t0 = time.time()

def jaffe(image, c=0.15, depth=3.0, seed=None):
    rng = np.random.RandomState(seed)
    img = image.astype(np.float32) / 255.0
    att = np.exp(-c * depth)
    bs = np.zeros_like(img)
    bs[:,:,0] = 0.10 * (1 - att)
    bs[:,:,1] = 0.30 * (1 - att)
    bs[:,:,2] = 0.05 * (1 - att)
    t = img * att + bs
    n = rng.normal(0, 0.02 * c * 10, img.shape).astype(np.float32)
    return (np.clip(t + n, 0, 1) * 255).astype(np.uint8)

LEVELS = {"light": 0.05, "medium": 0.15, "heavy": 0.30}
TRAIN_IMG = Path("data/processed/train/images")
TRAIN_LBL = Path("data/processed/train/labels")
TEST_IMG = Path("data/processed/test/images")
TEST_LBL = Path("data/processed/test/labels")

# Augment TRAINING
train_imgs = sorted(TRAIN_IMG.glob("*.jpg"))
print(f"Augmenting {len(train_imgs)} train images x 3 levels...")
for lname, cval in LEVELS.items():
    oimg = Path(f"data/augmented_train/{lname}/images"); oimg.mkdir(parents=True, exist_ok=True)
    olbl = Path(f"data/augmented_train/{lname}/labels"); olbl.mkdir(parents=True, exist_ok=True)
    for i, ip in enumerate(tqdm(train_imgs, desc=f"train-{lname}")):
        img = cv2.imread(str(ip))
        if img is None: continue
        cv2.imwrite(str(oimg / f"{lname}_{ip.name}"), jaffe(img, c=cval, seed=42+i))
        ls = TRAIN_LBL / f"{ip.stem}.txt"
        if ls.exists(): shutil.copy2(ls, olbl / f"{lname}_{ip.stem}.txt")

# Augment TEST
test_imgs = sorted(TEST_IMG.glob("*.jpg"))
print(f"\nAugmenting {len(test_imgs)} test images x 3 levels...")
for lname, cval in LEVELS.items():
    oimg = Path(f"data/augmented_test/{lname}/images"); oimg.mkdir(parents=True, exist_ok=True)
    olbl = Path(f"data/augmented_test/{lname}/labels"); olbl.mkdir(parents=True, exist_ok=True)
    for i, ip in enumerate(tqdm(test_imgs, desc=f"test-{lname}")):
        img = cv2.imread(str(ip))
        if img is None: continue
        cv2.imwrite(str(oimg / ip.name), jaffe(img, c=cval, seed=42+i))
        ls = TEST_LBL / f"{ip.stem}.txt"
        if ls.exists(): shutil.copy2(ls, olbl / ls.name)

print(f"\nDone ({time.time()-t0:.1f}s)")

Augmenting 5723 train images x 3 levels...


train-heavy: 100%|██████████| 5723/5723 [04:13<00:00, 22.59it/s]



Augmenting 1144 test images x 3 levels...


test-heavy: 100%|██████████| 1144/1144 [00:50<00:00, 22.67it/s]


Done (3556.6s)


In [4]:
import os
from pathlib import Path

CIMG = Path("data/combined_train/images"); CIMG.mkdir(parents=True, exist_ok=True)
CLBL = Path("data/combined_train/labels"); CLBL.mkdir(parents=True, exist_ok=True)

# Clean
c = 0
for f in Path("data/processed/train/images").glob("*.jpg"):
    d = CIMG / f.name
    if not d.exists(): os.symlink(f.resolve(), d); c += 1
for f in Path("data/processed/train/labels").glob("*.txt"):
    d = CLBL / f.name
    if not d.exists(): os.symlink(f.resolve(), d)

# Augmented
for level in ["light", "medium", "heavy"]:
    for f in Path(f"data/augmented_train/{level}/images").glob("*.jpg"):
        d = CIMG / f.name
        if not d.exists(): os.symlink(f.resolve(), d)
    for f in Path(f"data/augmented_train/{level}/labels").glob("*.txt"):
        d = CLBL / f.name
        if not d.exists(): os.symlink(f.resolve(), d)

total = len(list(CIMG.glob("*.jpg")))
print(f"Combined: {total} images ({total/c:.1f}x clean)")

Combined: 22892 images (4.0x clean)


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import yaml, os
from pathlib import Path
os.makedirs("configs/datasets", exist_ok=True)

with open("configs/datasets/trash_icra19_clean.yaml", "w") as f:
    yaml.dump({"path": str(Path("data/processed").resolve()), "train": "train/images",
               "val": "val/images", "test": "test/images", "names": {0: "plastic", 1: "bio"}}, f)

with open("configs/datasets/trash_icra19_augmented.yaml", "w") as f:
    yaml.dump({"path": str(Path("data").resolve()), "train": "combined_train/images",
               "val": "processed/val/images", "test": "processed/test/images",
               "names": {0: "plastic", 1: "bio"}}, f)

for level in ["light", "medium", "heavy"]:
    with open(f"configs/datasets/trash_icra19_turbid_{level}.yaml", "w") as f:
        yaml.dump({"path": str(Path(f"data/augmented_test/{level}").resolve()),
                   "train": "images", "val": "images", "test": "images",
                   "names": {0: "plastic", 1: "bio"}}, f)

print("Configs created:")
for f in sorted(Path("configs/datasets").glob("*.yaml")): print(f"  {f.name}")

Configs created:
  trash_icra19_augmented.yaml
  trash_icra19_clean.yaml
  trash_icra19_turbid_heavy.yaml
  trash_icra19_turbid_light.yaml
  trash_icra19_turbid_medium.yaml


In [22]:
import importlib
import inspect
import re
import subprocess, sys
from pathlib import Path

# Pin Ultralytics version for reproducibility
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.3.40"], check=True)

import torch
import ultralytics
from ultralytics import RTDETR

UL_ROOT = Path(ultralytics.__file__).resolve().parent


def _insert_after_imports(src_lines: list[str], snippet: str) -> list[str]:
    insert_idx = 0
    for i, line in enumerate(src_lines):
        if line.startswith("import ") or line.startswith("from "):
            insert_idx = i + 1
        if line.startswith("class "):
            break
    snippet_lines = snippet.rstrip("\n").splitlines(True)
    if snippet_lines and not snippet_lines[-1].endswith("\n"):
        snippet_lines[-1] += "\n"
    return src_lines[:insert_idx] + snippet_lines + src_lines[insert_idx:]


def _read_lines(p: Path) -> list[str]:
    return p.read_text(encoding="utf-8", errors="ignore").splitlines(True)


def _write_lines(p: Path, lines: list[str]) -> None:
    p.write_text("".join(lines), encoding="utf-8")


def _find_block_end(lines: list[str], start_i: int, next_pat: re.Pattern) -> int:
    for j in range(start_i + 1, len(lines)):
        if next_pat.match(lines[j]):
            return j
    return len(lines)


def _find_method_bounds(lines: list[str], class_start: int, class_end: int, method_name: str):
    def_pat = re.compile(rf"^(\s*)def\s+{re.escape(method_name)}\s*\(")
    def_i = None
    def_indent = None

    for i in range(class_start, class_end):
        m = def_pat.match(lines[i])
        if m:
            def_i = i
            def_indent = m.group(1)
            break

    if def_i is None:
        return None

    body_indent = def_indent + "    "
    body_start = def_i + 1

    next_def_pat = re.compile(rf"^{re.escape(def_indent)}def\s+")
    next_class_pat = re.compile(r"^class\s+")

    end_i = class_end
    for j in range(def_i + 1, class_end):
        if next_class_pat.match(lines[j]) or next_def_pat.match(lines[j]):
            end_i = j
            break

    return def_i, body_start, end_i, def_indent, body_indent


# Build a model once so we can locate the exact class/file Ultralytics uses in this version.
probe = RTDETR("rtdetr-l.pt")

candidates = []
for name, mod in probe.model.named_modules():
    if hasattr(mod, "input_proj"):
        ip = getattr(mod, "input_proj")
        if isinstance(ip, torch.nn.ModuleList):
            candidates.append((name, mod))

if not candidates:
    raise RuntimeError("No modules with input_proj (ModuleList) found in RTDETR model; patch target not found.")


def _score(n: str, cls_name: str) -> int:
    s = 0
    if "rtdetr" in (n.lower() + cls_name.lower()):
        s += 10
    if "decoder" in cls_name.lower():
        s += 5
    if "transform" in cls_name.lower():
        s += 3
    return s

candidates.sort(key=lambda t: _score(t[0], t[1].__class__.__name__), reverse=True)
TARGET_NAME, TARGET_MOD = candidates[0]
TARGET_CLASS_NAME = TARGET_MOD.__class__.__name__

TARGET_FILE = Path(inspect.getsourcefile(TARGET_MOD.__class__)).resolve()
if UL_ROOT not in TARGET_FILE.parents:
    raise RuntimeError(f"Refusing to patch unexpected file outside ultralytics package: {TARGET_FILE}")

print(f"Target class: {TARGET_CLASS_NAME} (module path in model: {TARGET_NAME})")
print(f"File: {TARGET_FILE}")

PATCHED_SIMAM_FILE = TARGET_FILE


# Define SimAM in the runtime (also used by the RTDETR init hook below)
class SimAM(torch.nn.Module):
    def __init__(self, lam=1e-4):
        super().__init__()
        self.lam = lam

    def forward(self, x):
        b, c, h, w = x.size()
        n = h * w - 1
        mu = x.mean(dim=[2, 3], keepdim=True)
        var = ((x - mu) ** 2).sum(dim=[2, 3], keepdim=True) / n
        e = (x - mu) ** 2 / (4 * (var + self.lam)) + 0.5
        return x * torch.sigmoid(e)


# --- Source patch (exportable) ---
lines = _read_lines(TARGET_FILE)

# Ensure SimAM exists in file
if not any(re.match(r"^class\s+SimAM\b", l) for l in lines):
    simam_class = inspect.getsource(SimAM) + "\n"
    simam_class = simam_class.replace("torch.nn.Module", "nn.Module")
    simam_class = "# === TURB-DETR SimAM (Yang et al. 2021) ===\n" + simam_class + "# === END SimAM ===\n"
    lines = _insert_after_imports(lines, simam_class)
    print("Injected SimAM class into file")

# Patch target class block (scoped)
class_pat = re.compile(rf"^class\s+{re.escape(TARGET_CLASS_NAME)}\b")
class_i = next((i for i, l in enumerate(lines) if class_pat.match(l)), None)
if class_i is None:
    raise RuntimeError(f"Couldn't find class {TARGET_CLASS_NAME} in {TARGET_FILE}")

class_end = _find_block_end(lines, class_i, re.compile(r"^class\s+"))

init_bounds = _find_method_bounds(lines, class_i, class_end, "__init__")
fwd_bounds = _find_method_bounds(lines, class_i, class_end, "forward")
if init_bounds is None or fwd_bounds is None:
    raise RuntimeError(f"Couldn't find __init__/forward in {TARGET_CLASS_NAME}")

init_def_i, init_body_start, init_body_end, _, init_body_indent = init_bounds
fwd_def_i, fwd_body_start, fwd_body_end, _, fwd_body_indent = fwd_bounds

# Ensure self.simam in __init__ body
if "self.simam" not in "".join(lines[init_body_start:init_body_end]):
    super_i = next((i for i in range(init_body_start, init_body_end) if "super().__init__()" in lines[i]), None)
    if super_i is None:
        raise RuntimeError(f"Couldn't find super().__init__() in {TARGET_CLASS_NAME}.__init__")
    lines.insert(super_i + 1, init_body_indent + "self.simam = SimAM()\n")
    print("Injected self.simam into file __init__")

# Ensure forward applies SimAM safely even if __init__ was bypassed (e.g., pickled .pt load)
fwd_body = "".join(lines[fwd_body_start:fwd_body_end])
need_guard = "hasattr(self, 'simam')" not in fwd_body and "hasattr(self, \"simam\")" not in fwd_body

if "# SimAM" not in fwd_body:
    m = re.match(r"^\s*def\s+forward\(\s*self\s*,\s*([A-Za-z_][A-Za-z0-9_]*)", lines[fwd_def_i])
    arg1 = m.group(1) if m else "x"

    insert_at = fwd_body_start
    if insert_at < fwd_body_end and lines[insert_at].lstrip().startswith(('"""', "'''")):
        q = lines[insert_at].lstrip()[:3]
        j = insert_at
        if lines[j].count(q) >= 2:
            insert_at = j + 1
        else:
            j += 1
            while j < fwd_body_end and q not in lines[j]:
                j += 1
            insert_at = min(j + 1, fwd_body_end)

    guard_line = fwd_body_indent + "if not hasattr(self, 'simam'): self.simam = SimAM()  # SimAM\n"
    apply_line = (
        fwd_body_indent
        + f"{arg1} = [self.simam(t) if getattr(t, 'ndim', 0) == 4 else t for t in {arg1}] "
        + f"if isinstance({arg1}, (list, tuple)) else {arg1}  # SimAM\n"
    )
    lines.insert(insert_at, guard_line)
    lines.insert(insert_at + 1, apply_line)
    print("Injected forward SimAM guard+apply lines into file")

elif need_guard:
    # We already injected a SimAM apply line earlier; add a guard right above the first SimAM-marked line
    first_simam_i = None
    for i in range(fwd_body_start, fwd_body_end):
        if "# SimAM" in lines[i]:
            first_simam_i = i
            break
    if first_simam_i is not None:
        lines.insert(first_simam_i, fwd_body_indent + "if not hasattr(self, 'simam'): self.simam = SimAM()  # SimAM\n")
        print("Inserted missing forward guard line into file")

_write_lines(TARGET_FILE, lines)


# --- Optional runtime reinforcement: attach SimAM after RTDETR loads ---
if not getattr(RTDETR, "__simam_init_hook__", False):
    _orig_rtdetr_init = RTDETR.__init__

    def _patched_rtdetr_init(self, *args, **kwargs):
        _orig_rtdetr_init(self, *args, **kwargs)
        for m in self.model.modules():
            if m.__class__.__name__ == TARGET_CLASS_NAME and not hasattr(m, "simam"):
                m.simam = SimAM()

    RTDETR.__init__ = _patched_rtdetr_init
    RTDETR.__simam_init_hook__ = True
    print("Patched RTDETR.__init__ to attach SimAM after loading")
else:
    print("RTDETR.__init__ hook already patched")


# Sanity check
check = RTDETR("rtdetr-l.pt")
decoders = [m for m in check.model.modules() if m.__class__.__name__ == TARGET_CLASS_NAME]
print(f"{TARGET_CLASS_NAME} instances in model: {len(decoders)}")
print(f"Decoder[0] has simam: {hasattr(decoders[0], 'simam') if decoders else False}")

simam_count = sum(1 for m in check.model.modules() if m.__class__.__name__ == "SimAM")
print(f"SimAM modules in model: {simam_count}")
assert decoders and hasattr(decoders[0], "simam"), "SimAM not attached to target module"
assert simam_count > 0, "SimAM module not present in model"


Target class: RTDETRDecoder (module path in model: model.28)
File: /usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/head.py
Inserted missing forward guard line into file
RTDETR.__init__ hook already patched
RTDETRDecoder instances in model: 1
Decoder[0] has simam: True
SimAM modules in model: 1


In [20]:
from ultralytics import RTDETR

model = RTDETR("rtdetr-l.pt")

total_params = sum(p.numel() for p in model.model.parameters())
simam_modules = [m for m in model.model.modules() if m.__class__.__name__ == "SimAM"]
simam_params = sum(p.numel() for m in simam_modules for p in m.parameters())

print(f"Parameters (total): {total_params:,}")
print(f"SimAM modules found: {len(simam_modules)}")
print(f"SimAM learnable params: {simam_params:,}")

assert len(simam_modules) > 0, "SimAM module not found in the model (patch not active)"
assert simam_params == 0, "SimAM should add zero learnable parameters"
print("ZERO parameter increase from SimAM confirmed")


Parameters (total): 32,970,476
SimAM modules found: 1
SimAM learnable params: 0
ZERO parameter increase from SimAM confirmed


In [23]:
import shutil
from pathlib import Path

SAVE = Path("/content/drive/MyDrive/turb_detr_results/week2_prep")
SAVE.mkdir(parents=True, exist_ok=True)

# Save the actual file we patched (HybridEncoder lives in different files across Ultralytics versions)
patched_file = globals().get("PATCHED_SIMAM_FILE", None)
if patched_file is not None:
    patched_file = Path(patched_file)
    if patched_file.exists():
        shutil.copy2(patched_file, SAVE / patched_file.name)
        print(f"Saved patched Ultralytics file: {patched_file.name}")
    else:
        print(f"WARNING: patched file not found on disk: {patched_file}")
else:
    print("WARNING: PATCHED_SIMAM_FILE not set (did the SimAM patch cell run?)")

# Save dataset configs
for f in Path("configs/datasets").glob("*.yaml"):
    shutil.copy2(f, SAVE / f.name)

print(f"Saved to: {SAVE}")
print("Now run Week 2B training notebook (GPU required)")


Saved patched Ultralytics file: head.py
Saved to: /content/drive/MyDrive/turb_detr_results/week2_prep
Now run Week 2B training notebook (GPU required)
